In [245]:
from lib import Env
from chula_rl.alphazero.santorini.santorinigo.fast import Santorini
import random
from numba import njit
import time
from datetime import datetime
import numpy as np

In [269]:
env = Env(False)
ret1 = env.reset(1)
ret2 = env.reset(2)

In [270]:
cnt = 0
while True:
    s, r, done, info, legal_moves = ret1.wait()
    print(s,r,done,legal_moves,sep='\n')
    if done:
        print('player 1 reward:', r)
        cnt += 1
        break
    else:
#         a = random.choice(legal_moves)
        board, workers, parts, current_player, done = env.env.get_state().values()
        a, val = make_a_move(board, workers, parts)
        ret1 = env.step(a, 1)
    
    print('make move a={}, val={}'.format(a, val))
    print('-------------------------------')        
    s, r, done, info, legal_moves = ret2.wait()
    print(s,r,done,legal_moves,sep='\n')
    if done:
        print('player 2 reward:', r)
        cnt += 1
        break
    else:
        a = random.choice(legal_moves)
        ret2 = env.step(a, 2)
    print('make move a=',a)
    print('-------------------------------')        


[[[ 0  0  0  0  0]
  [ 0  0  0  0  0]
  [ 0  0  0  0  0]
  [ 0  0  0  0  0]
  [ 0  0  0  0  0]]

 [[ 0  0 -1  0  0]
  [ 0  0  0  0  0]
  [ 1  0  0  0  2]
  [ 0  0  0  0  0]
  [ 0  0 -2  0  0]]

 [[ 0  0  0  0  0]
  [ 0 22  0  0  0]
  [ 0  0 18  0  0]
  [ 0  0  0 14  0]
  [ 0  0  0  0 18]]]
None
None
[27, 28, 29, 30, 31, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 96, 97, 98, 99, 100]
Time elapsed: 0:00:00.326110
a_idx=54,val=2.8000000000000003
cnt = 533
make move a=54, val=2.8000000000000003
-------------------------------
[[[ 0  0  0  0  0]
  [ 0  0  0  0  0]
  [ 0  0  1  0  0]
  [ 0  0  0  0  0]
  [ 0  0  0  0  0]]

 [[ 0  0  0  0  0]
  [ 0  0  1  0  0]
  [-1  0  0  0 -2]
  [ 0  0  0  0  0]
  [ 0  0  2  0  0]]

 [[ 0  0  0  0  0]
  [ 0 21  0  0  0]
  [ 0  0 18  0  0]
  [ 0  0  0 14  0]
  [ 0  0  0  0 18]]]
-0.0
Fal

Time elapsed: 0:00:00.312081
a_idx=44,val=29.000000000000007
cnt = 485
make move a=44, val=29.000000000000007
-------------------------------
[[[ 0  1  0  0  0]
  [ 0  1  1  1  0]
  [ 0  0  2  2  1]
  [ 0  0  2  2  0]
  [ 0  0  0  0  0]]

 [[ 0  0  0  0  0]
  [-1  0 -2  0  0]
  [ 0  0  2  0  0]
  [ 0  0  1  0  0]
  [ 0  0  0  0  0]]

 [[ 0  0  0  0  0]
  [ 0 13  0  0  0]
  [ 0  0 14  0  0]
  [ 0  0  0 14  0]
  [ 0  0  0  0 18]]]
-0.0
False
[12, 14, 15, 19, 20, 21, 22, 32, 33, 34, 35, 37, 38, 49, 50, 52, 54, 55, 56, 57, 59, 61, 62, 67, 68, 70, 71, 75, 76, 77, 78, 79, 83, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 96, 97, 98, 99, 100, 102, 103, 105, 106, 107, 109, 110, 120, 121, 122, 124, 126, 127]
make move a= 122
-------------------------------
[[[ 0  1  0  0  0]
  [ 0  1  1  1  1]
  [ 0  0  2  2  1]
  [ 0  0  2  2  0]
  [ 0  0  0  0  0]]

 [[ 0  0  0  0  0]
  [ 1  0  0  0  0]
  [ 0  0 -2  2  0]
  [ 0  0 -1  0  0]
  [ 0  0  0  0  0]]

 [[ 0  0  0  0  0]
  [ 0 12  0  0  0]
  [ 0  0 14  0  

In [267]:
def get_board_value(env):
    board, workers, parts, current_player, done = env.get_state().values()
    score = board[workers[0,0], workers[0,1]]*board[workers[0,0], workers[0,1]] + board[workers[1,0],workers[1,1]]*board[workers[1,0],workers[1,1]]
    x,y = np.where(board != 0)
    for _x, _y in zip(x,y):
        score += 0.1 * board[_x,_y] *(20 - abs(workers[0,0] - _x) - abs(workers[0,1] - _y) - abs(workers[1,0] - _x) - abs(workers[1,1] - _y))                
    score -= board[workers[2,0], workers[2,1]] + board[workers[3,0],workers[3,1]]
    return score
cnt = 0
new_env = Santorini()
def alphabeta(env, depth, alpha, beta, isMaximizing):
    global cnt, new_env
    cnt+= 1
    board, workers, parts, current_player, done = env.get_state().values()
#     print('------------------------------------------------------------')
#     print('depth:{}, alpha:{}, beta:{}, isMaximizing:{}'.format(depth,alpha,beta,isMaximizing))
#     print(board, workers,parts,current_player, done)
#     print('------------------------------------------------------------')
    if done:
        if current_player == 1: #player 1 has won
            return -1, np.inf
        else:
#             print(board,workers,current_player)
            return -1, -np.inf #player 2 has won
    if depth == 0:
        return -1, get_board_value(env)
    
    legal_moves = env.legal_moves()
    
    if isMaximizing:
#         print('AAAAA')
        value = -np.inf
        select_a = legal_moves[0]
        del env
#         print('@', legal_moves)
        for a_idx in legal_moves:
            new_env = Santorini()
            new_env.reset()
            new_env.set_state(np.array(board), np.array(workers), np.array(parts), current_player, done)
#             print(board)
#             print(new_env.legal_moves())
#             print('*',new_env.get_state())
            new_env.step(a_idx)
#             print(board)
#             print('!',new_env.get_state())
            _, val = alphabeta(new_env, depth-1, alpha, beta, False)
#             print('=',_,val)
            if val > value:
                alpha = val
                value = val
                select_a = a_idx
            if alpha >= beta: #no need to search anymore
                break
        return select_a, value
    else:
#         print('BBBBB')
        value = np.inf
        select_a = legal_moves[0]
#         print('#',legal_moves)
        del env
        for a_idx in legal_moves:
            new_env = Santorini()
            new_env.reset()
            new_env.set_state(np.array(board), np.array(workers), np.array(parts), current_player, done)
#             print('$', new_env.legal_moves())
#             print('b',new_env.get_state())
            new_env.step(a_idx)
#             print('a',new_env.get_state())
            _, val = alphabeta(new_env, depth-1, alpha, beta, True)
            if val < value:
                value = val
                beta = val
                select_a = a_idx
            if alpha >= beta: #no need to search anymore
                break
        return select_a, value
    
def make_a_move(board, workers, parts):
    depth = 2
    global new_env;
    new_env.reset()
    global cnt
    cnt = 0
#     board, workers, parts, current_player, done = env.get_state().values()
    new_env.set_state(np.array(board), np.array(workers), np.array(parts), -1, False)
    t = datetime.now()
    a_idx, val = alphabeta(new_env, depth, -np.inf, np.inf, True)
    print('Time elapsed:',datetime.now()-t)
    print('a_idx={},val={}'.format(a_idx, val))
    print('cnt =',cnt)
    return a_idx, val

In [268]:
env_test.reset()
print(env_test.get_state())
board, workers, parts, current_player, done = env_test.get_state().values()
make_a_move(board,workers,parts)

{'board': array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]]), 'workers': array([[0, 2],
       [4, 2],
       [2, 0],
       [2, 4]]), 'parts': array([ 0, 22, 18, 14, 18]), 'current_player': -1, 'done': False}
Time elapsed: 0:00:00.311592
a_idx=54,val=2.8000000000000003
cnt = 533


(54, 2.8000000000000003)

In [212]:
get_board_value(env_test)

0

In [129]:
board, workers, parts, current_player, done = env_test.get_state().values()
board

array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

In [216]:
env.env.get_state()

{'board': array([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]]), 'workers': array([[0, 2],
        [4, 2],
        [2, 0],
        [2, 4]]), 'parts': array([ 0, 22, 18, 14, 18]), 'current_player': -1, 'done': False}

In [229]:
s = new_env.reset()
s

array([[[ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0]],

       [[ 0,  0, -1,  0,  0],
        [ 0,  0,  0,  0,  0],
        [ 1,  0,  0,  0,  2],
        [ 0,  0,  0,  0,  0],
        [ 0,  0, -2,  0,  0]],

       [[ 0,  0,  0,  0,  0],
        [ 0, 22,  0,  0,  0],
        [ 0,  0, 18,  0,  0],
        [ 0,  0,  0, 14,  0],
        [ 0,  0,  0,  0, 18]]])

In [230]:
s[0]

array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

In [231]:
s[1]

array([[ 0,  0, -1,  0,  0],
       [ 0,  0,  0,  0,  0],
       [ 1,  0,  0,  0,  2],
       [ 0,  0,  0,  0,  0],
       [ 0,  0, -2,  0,  0]])

In [240]:
np.array([np.array(np.where(s[1] == -1)).reshape(-1),
np.array(np.where(s[1] == -2)).reshape(-1),
np.array(np.where(s[1] == 1)).reshape(-1),
np.array(np.where(s[1] == 2)).reshape(-1)])

array([[0, 2],
       [4, 2],
       [2, 0],
       [2, 4]])

In [244]:
s[2].diagonal()

array([ 0, 22, 18, 14, 18])

In [258]:
s = np.array([[ 2 ,3,0 ,0 ,0],
  [ 0 ,0,0 ,0 ,0],
  [ 1, 1, 0, 0, 0],
  [ 2, 2, 1, 0, 0],
  [ 0, 0, 0, 0, 0]])
s

array([[2, 3, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0],
       [2, 2, 1, 0, 0],
       [0, 0, 0, 0, 0]])

In [262]:
np.where(s != 0)

(array([0, 0, 2, 2, 3, 3, 3]), array([0, 1, 0, 1, 0, 1, 2]))

In [272]:
env_test.reset()
board, workers, parts, current_player, done = env_test.get_state().values()
env_test.set_state(board,workers,parts,current_player,done)
print(get_board_value(env_test))
# make_a_move(board,workers,parts)

0
